# EPRBench 可视化推理 Notebook

本 notebook 用于加载 SGVPR 模型权重，对 EventVPR 测试集中的任意 query 进行图像地点识别推理，并直观展示：

- 输入 query 图像
- 模型检索到的 Top-K 候选图像
- GT（Ground Truth）正样本标注
- 对应场景文本描述
- 当前 query 的 Recall@N 结果

运行前请确保：
1. 已激活项目环境 (`conda activate myenv` 或对应环境)
2. `model/SGVPR.pth` 与 `model/dinov2_vitb14_pretrain.pth` 已存在
3. `data/EventVPR/` 数据集已准备就绪


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import torch
import faiss
import matplotlib.pyplot as plt
from PIL import Image
from omegaconf import OmegaConf

# Resolve project root. In Jupyter __file__ is undefined, so fall back to cwd.
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

# If running from notebooks/ directory, go up one level
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)

# Ensure src is importable
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import rootutils
rootutils.setup_root(str(PROJECT_ROOT), indicator=".project-root", pythonpath=True)

from src.models.eprbench_module import EPRBenchLitModule
from src.data.eprbench_datamodule import EPRBenchDataModule
from src.data.components.eventvpr_dataset import EventVPRDataset

# Prefer GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 1. 配置

通过 OmegaConf 手动拼装与 `configs/eval.yaml` 等价的配置对象。你可以在这里修改数据集路径、checkpoint 路径或 recall 设置。

In [ ]:
# --------------------------- User-configurable ---------------------------
DATASET_FOLDER = Path("data/EventVPR")            # 数据集根目录
CKPT_PATH = Path("model/SGVPR.pth")                # SGVPR 权重路径
FOUNDATION_PATH = Path("model/dinov2_vitb14_pretrain.pth")
USE_TEXT = True
TEXT_FOLDER = None                                 # None -> DATASET_FOLDER/scene_descriptions
FEATURES_DIM = 10752                               # 14 * 768
RECALL_VALUES = [1, 5, 10, 100]
INFER_BATCH_SIZE = 18
NUM_WORKERS = 4
TOP_K = 10                                         # 可视化检索数量
# -------------------------------------------------------------------------

# Compose the same config tree used by Hydra
cfg = OmegaConf.create({
    "data": {
        "_target_": "src.data.eprbench_datamodule.EPRBenchDataModule",
        "dataset_folder": str(DATASET_FOLDER),
        "train_batch_size": 18,
        "infer_batch_size": INFER_BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": True,
        "use_text": USE_TEXT,
        "text_folder": TEXT_FOLDER,
        "img_per_place": 5,
        "min_img_per_place": 5,
        "resize": [224, 224],
    },
    "model": {
        "_target_": "src.models.eprbench_module.EPRBenchLitModule",
        "lambda_contrast": 0.15,
        "temperature": 0.07,
        "use_text": USE_TEXT,
        "features_dim": FEATURES_DIM,
        "test_method": "hard_resize",
        "recall_values": RECALL_VALUES,
        "infer_batch_size": INFER_BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pca_dim": None,
        "efficient_ram_testing": False,
        "majority_weight": 0.01,
        "compile": False,
        "use_freq_contrastive_loss": False,
        "freq_loss_weight": 0.0,
        "freq_num_negatives": 2,
        "use_temporal_consistency": False,
        "temporal_consistency_weight": 0.05,
        "text_dropout_prob": 0.0,
        "word_dropout_prob": 0.0,
        "sacf_residual_loss_weight": 0.0,
        "sacf_noise_loss_weight": 0.0,
        "optimizer": {
            "_target_": "torch.optim.AdamW",
            "_partial_": True,
            "lr": 0.0001,
            "weight_decay": 0.001,
        },
        "scheduler": {
            "_target_": "torch.optim.lr_scheduler.StepLR",
            "_partial_": True,
            "gamma": 0.5,
        },
        "net": {
            "_target_": "src.models.components.network.SGVPRNet",
            "pretrained_foundation": True,
            "foundation_model_path": str(FOUNDATION_PATH),
            "use_frequency_enhance": False,
            "use_freq_topk": False,
            "freq_score_weight": 0.10,
            "freq_low_cut": 0.25,
            "freq_mid_cut": 0.60,
            "freq_highpass_cut": 0.25,
            "freq_max_gain": 0.50,
            "freq_gamma_init": 0.0,
            "mods": {
                "cross_spectral": {"enabled": False, "layer_idx": 6},
                "sacf": {"enabled": False, "variant": "original", "layer_idx": 6},
                "scfm": {"enabled": False},
                "igr": {"enabled": False},
                "lgfp": {"enabled": False},
                "fbgw": {"enabled": False},
                "bim": {"enabled": False},
                "spm": {"enabled": False},
                "event_cues": {"enabled": False},
                "event_aware_token_selector": {"enabled": False},
                "adaptive_text_topk": {"enabled": False},
                "temporal_fusion": {"enabled": False},
            },
        },
    },
})

print("Configuration ready.")


## 2. 加载数据集

In [ ]:
import hydra

datamodule: EPRBenchDataModule = hydra.utils.instantiate(cfg.data)
datamodule.setup("test")
test_dataset: EventVPRDataset = datamodule.data_test
print(f"Test dataset size: {len(test_dataset)}")
print(f"Database images: {test_dataset.database_num}")
print(f"Query images: {test_dataset.queries_num}")
print(f"Recall values: {RECALL_VALUES}")


## 3. 构建模型并加载权重

`model/SGVPR.pth` 是官方训练脚本保存的字典，包含 `model_state_dict` 与 `text_encoder_state_dict`。我们手动加载到 LightningModule 中。

In [ ]:
model: EPRBenchLitModule = hydra.utils.instantiate(cfg.model)
model = model.to(device)
model.eval()

# Load checkpoint
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
print("Checkpoint keys:", list(ckpt.keys()))

model_state = ckpt["model_state_dict"]
# Strip possible 'module.' prefix from state_dict keys
model_state = {k.replace("module.", ""): v for k, v in model_state.items()}

missing, unexpected = model.net.load_state_dict(model_state, strict=False)
print(f"Net missing keys: {len(missing)}, unexpected: {len(unexpected)}")
if missing:
    print("Missing:", missing[:10])
if unexpected:
    print("Unexpected:", unexpected[:10])

# Text encoder weights (if present and shapes match)
if "text_encoder_state_dict" in ckpt:
    te_missing, te_unexpected = model.text_encoder.load_state_dict(
        ckpt["text_encoder_state_dict"], strict=False
    )
    print(f"Text encoder missing: {len(te_missing)}, unexpected: {len(te_unexpected)}")

print("Model loaded successfully.")


## 4. 提取 Database 与 Query 特征并构建 FAISS 索引

In [ ]:
def extract_features(dataloader):
    """Extract normalized descriptors for every image in the dataloader."""
    all_features = []
    all_indices = []
    with torch.no_grad():
        for batch in dataloader:
            if len(batch) == 3:
                imgs, indices, text_desc = batch
                text_vec = model.text_encoder(text_desc)
                text_tokens, text_mask = model.text_encoder.encode_batch_tokens(text_desc)
                text_tokens = text_tokens.to(device)
                text_mask = text_mask.to(device)
                feats = model.net(
                    imgs.to(device),
                    text=text_vec,
                    text_tokens=text_tokens,
                    text_mask=text_mask,
                )
            else:
                imgs, indices = batch
                feats = model.net(imgs.to(device))
            all_features.append(feats.cpu())
            all_indices.append(indices.cpu())
    features = torch.cat(all_features, dim=0).numpy()
    indices = torch.cat(all_indices, dim=0).numpy()
    return features, indices

# Extract all features
test_loader = datamodule.test_dataloader()
all_features, all_indices = extract_features(test_loader)

# Sort by global index
sorted_features = np.empty((len(test_dataset), FEATURES_DIM), dtype=np.float32)
sorted_features[all_indices] = all_features

database_features = sorted_features[: test_dataset.database_num]
queries_features = sorted_features[test_dataset.database_num :]

# Build FAISS index
faiss_index = faiss.IndexFlatL2(FEATURES_DIM)
faiss_index.add(database_features)

# Run retrieval
top_k_max = max(max(RECALL_VALUES), TOP_K)
distances, predictions = faiss_index.search(queries_features, top_k_max)

# Compute recalls per query and overall
positives_per_query = test_dataset.get_positives()
recalls_per_query = np.zeros((len(positives_per_query), len(RECALL_VALUES)), dtype=bool)
for q_idx, (pred, pos) in enumerate(zip(predictions, positives_per_query)):
    for r_i, n in enumerate(RECALL_VALUES):
        recalls_per_query[q_idx, r_i] = np.any(np.isin(pred[:n], pos))

overall_recalls = recalls_per_query.mean(axis=0) * 100
print("Overall recalls:")
for n, rec in zip(RECALL_VALUES, overall_recalls):
    print(f"  R@{n}: {rec:.2f}%")


## 5. 可视化辅助函数

In [ ]:
def load_raw_image(path, resize=(224, 224)):
    """Load a raw RGB image for visualization (not normalized)."""
    img = Image.open(path).convert("RGB")
    img = img.resize(resize[::-1])  # PIL uses (W, H)
    return np.array(img)


def visualize_query_retrieval(query_idx, top_k=TOP_K, figsize=(16, 4)):
    """Visualize one query and its retrieved top-k database images."""
    query_global_idx = test_dataset.database_num + query_idx
    query_path = test_dataset.all_images_paths[query_global_idx]
    query_place_id = test_dataset._all_place_ids[query_global_idx]
    positives = positives_per_query[query_idx]

    pred = predictions[query_idx][:top_k]

    # Determine per-recall hits
    hits = {n: np.any(np.isin(pred[:n], positives)) for n in RECALL_VALUES}

    n_cols = top_k + 1
    fig, axes = plt.subplots(1, n_cols, figsize=figsize)
    if n_cols == 1:
        axes = [axes]

    # Query image
    axes[0].imshow(load_raw_image(query_path))
    axes[0].set_title(f"Query #{query_idx}\nPlace: {query_place_id}")
    axes[0].axis("off")

    # Text description (if available)
    text_desc = test_dataset.text_descriptions.get(query_path, "")
    if text_desc:
        axes[0].text(
            0.5, -0.12, text_desc[:120] + ("..." if len(text_desc) > 120 else ""),
            transform=axes[0].transAxes,
            ha="center", va="top", fontsize=8, wrap=True,
        )

    # Retrieved images
    for rank, db_idx in enumerate(pred, start=1):
        db_path = test_dataset.all_images_paths[db_idx]
        db_place_id = test_dataset._all_place_ids[db_idx]
        is_positive = db_idx in positives
        color = "green" if is_positive else "red"
        title_color = "white" if is_positive else "black"

        ax = axes[rank]
        ax.imshow(load_raw_image(db_path))
        gt_label = "GT✓" if is_positive else "GT✗"
        ax.set_title(
            f"Rank {rank}\nPlace: {db_place_id}\n{gt_label}",
            color=title_color,
            backgroundcolor=color,
        )
        ax.axis("off")

    # Overall subtitle
    hit_str = ", ".join([f"R@{n}: {'✓' if hits[n] else '✗'}" for n in RECALL_VALUES])
    fig.suptitle(f"Retrieval results for query #{query_idx} | {hit_str}", y=1.02)
    plt.tight_layout()
    plt.show()

    return {
        "query_idx": query_idx,
        "query_path": query_path,
        "query_place_id": query_place_id,
        "predictions": pred,
        "positives": positives,
        "recall_hits": hits,
        "text": text_desc if text_desc else "",
    }


## 6. 运行可视化推理

修改 `query_idx` 或直接运行随机 query。绿色标题背景表示 GT 正样本，红色表示非正样本。

In [ ]:
# Pick a query index (0 .. queries_num-1)
query_idx = random.randint(0, test_dataset.queries_num - 1)
# query_idx = 0  # uncomment to fix a specific query

result = visualize_query_retrieval(query_idx, top_k=TOP_K)
print("Query path:", result["query_path"])
print("Predictions (top-{}):".format(TOP_K), result["predictions"])
print("GT positives:", result["positives"])
print("Text description:", result["text"][:200])


## 7. 批量查看成功/失败样例（可选）

下面随机抽取若干 query，直观展示模型在哪些样本上 R@1 成功、哪些失败。

In [ ]:
# Show 3 successful R@1 examples and 3 failures
success_indices = np.where(recalls_per_query[:, 0])[0]
failure_indices = np.where(~recalls_per_query[:, 0])[0]

print(f"R@1 successes: {len(success_indices)}, failures: {len(failure_indices)}")

for label, idx_pool in [("R@1 SUCCESS", success_indices), ("R@1 FAILURE", failure_indices)]:
    if len(idx_pool) == 0:
        continue
    samples = random.sample(list(idx_pool), min(3, len(idx_pool)))
    print(f"\n--- {label} examples ---")
    for q in samples:
        visualize_query_retrieval(q, top_k=TOP_K)
